# Mouse Embryo

Run this example in five steps. Set `SPADOT_RUN_NAME` for a new output folder and, if needed, `SPADOT_DEVICE` for your GPU. The input must contain raw counts, `obs["timepoint"]`, and `obsm["spatial"]`.

## 1. Load the data

In [ ]:
import os
import re
import subprocess
from pathlib import Path
import anndata as ad
import pandas as pd
from IPython.display import Image, display

examples = Path.cwd().resolve()
if not (examples / 'MouseEmbryo_data/MouseEmbryo.h5ad').is_file():
    examples = examples / "examples"
raw_file = examples / 'MouseEmbryo_data/MouseEmbryo.h5ad'
if not raw_file.is_file():
    raise FileNotFoundError(raw_file)
run_name = os.environ.get("SPADOT_RUN_NAME", "example_run")
if not re.fullmatch(r"[A-Za-z0-9_-]+", run_name):
    raise ValueError("SPADOT_RUN_NAME must contain letters, numbers, - or _")
run_dir = examples / "runs" / run_name / 'MouseEmbryo'
run_dir.mkdir(parents=True, exist_ok=True)
config_file = examples.parent / "SpaDOT" / "config.yaml"
data = ad.read_h5ad(raw_file, backed="r")
print(data.shape, data.obs["timepoint"].value_counts().sort_index().to_dict())
data.file.close()
print("Results:", run_dir)

## 2. Run SpaDOT preprocess

In [ ]:
input_file = run_dir / f"preprocessed_{raw_file.name}"
if input_file.exists():
    raise FileExistsError(f"{input_file} already exists. Choose a new SPADOT_RUN_NAME.")
subprocess.run([
    "SpaDOT", "preprocess", "--data", str(raw_file),
    "--output_dir", str(run_dir), "--n_features", '3000,3000,3000',
], check=True)
print("Preprocessed:", input_file)

## 3. Run SpaDOT train

In [ ]:
latent_file = run_dir / "latent.h5ad"
if latent_file.exists():
    raise FileExistsError(f"{latent_file} already exists. Choose a new SPADOT_RUN_NAME.")
env = dict(os.environ, PYTHONHASHSEED="2026", CUBLAS_WORKSPACE_CONFIG=":4096:8",
           OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
subprocess.run([
    "SpaDOT", "train", "--data", str(input_file), "--output_dir", str(run_dir),
    "--config", str(config_file), "--device", os.environ.get("SPADOT_DEVICE", "cuda:0"),
    "--save_model",
], check=True, env=env)
print("Latent:", latent_file)

## 4. Run SpaDOT analyze

In [ ]:
subprocess.run([
    "SpaDOT", "analyze", "--data", str(latent_file),
    "--output_dir", str(run_dir), "--n_clusters", '5,5,5',
], check=True)
print("Spatial plots and transition matrices:", run_dir)

## 5. Obtain the results

View the spatial domains and conditional transition scores produced by this run. If the raw data include annotations, compare them with the predicted domains using ARI.

In [ ]:
from sklearn.metrics import adjusted_rand_score

domains_file = run_dir / "domains.csv"
if not domains_file.is_file():
    raise FileNotFoundError(f"Run SpaDOT analyze first: {domains_file}")
domains = pd.read_csv(domains_file, dtype=str).set_index("spot")
data = ad.read_h5ad(raw_file, backed="r")
timepoints = ['0', '1', '2']
if "annotation" in data.obs:
    labels = data.obs["annotation"].astype(str)
    for tp in timepoints:
        part = domains[domains["timepoint"] == tp]
        print(tp, "ARI:", round(adjusted_rand_score(labels.loc[part.index], part["cluster"]), 3))
data.file.close()

for tp in timepoints:
    display(Image(filename=str(run_dir / f"{tp}_domains.png")))

mapping = run_dir / "domain_display_mapping.csv"
if mapping.is_file():
    print("Obtained domain ID → displayed name:")
    display(pd.read_csv(mapping, dtype=str)[["stage", "domain_id", "display_name"]])

for first, second in zip(range(len(timepoints) - 1), range(1, len(timepoints))):
    print(f"Transition {timepoints[first]} → {timepoints[second]}")
    display(Image(filename=str(run_dir / f"transition_dotplot_{first}_{second}.png")))
    print("Analyzed conditional transition-score matrix:")
    table = run_dir / f"transition_probability_min_ordered_{first}_{second}.csv"
    display(pd.read_csv(table, index_col=0))
